# Parte 1: Descarga de películas

En este código se encuentra el procedimiento seguido para obtener los registros válidos de películas, de los cuales podremos obtener el género y el póster que le corresponde.

In [43]:
#Importaciones necesarias
import requests
import random
import pandas as pd
from datetime import datetime
import os

In [44]:
#Función para indicar si el texto tiene comas
def tiene_comas(cadena):
    if isinstance(cadena, str) and ',' in cadena:
        return True
    return False

In [45]:
#Función para indicar si el texto es una fecha
def es_fecha(valor):
    try:
        datetime.strptime(valor, '%Y-%m-%d')
        return True
    except ValueError:
        return False

In [46]:
#Función para obtener el año de una fecha
def ObtenerAnio(valor):
    if es_fecha(valor):
        return datetime.strptime(valor, '%Y-%m-%d').year
    else:
        return valor[:4]

Hay 2 cosas que se estarán llenando con la información, uno es el dataframe *movies*, que irá acumulando los registros de películas que pasen los filtros aplicados. El segundo es *randoms*, el cual almacenará el id de la película para no almacenar repetidos o consultar doble un mismo registro. 

In [47]:
#Leemos los archivos
movies = pd.DataFrame()
randoms = pd.DataFrame()
lista_randoms = []

Por tema de recursos, este código se estuvo ejecutando varias veces en diferentes etapas, por lo que leemos tanto los randoms como las películas que ya se han almacenado en ejecuciones anteriores para continuar acumulando.

In [48]:
with open('C:/Users/clau_/OneDrive/Documentos/FCFM/03_Tetramestre/Aprendizaje_Profundo/PIA/lista_randoms.csv') as f:
    for line in f:
        lista_randoms.append(line.strip())

In [49]:
with open('C:/Users/clau_/OneDrive/Documentos/FCFM/03_Tetramestre/Aprendizaje_Profundo/PIA/movies.csv', encoding='utf-8') as f:
    movies = pd.read_csv(f, sep=',')

Los id de las películas constan de 7 posiciones numéricas, inicié con el id 9999999 y fui en retroceso en lotes de 200,000, 100,000 o 50,000 registros, dependiendo el tiempo que tuviera disponible para vigilar la ejecución del programa. Se fue en retroceso para garantizar que se iba a contar con películas a color y con títulos más recientes.

In [50]:
#Asignación de variables iniciales
count = 6374374       
movie_tries = count - 50000
movie_counter = 0

Para el ciclo donde se hizo la consulta a la página, se tomaron los siguientes filtros:
* Película creada a partir de 1970.
* Que sea una película.
* Que el género de la película no sea Adulto, Documental, Corto ni Biografía.
* Que la columna de Poster tenga información.
* Que la película tenga idioma inglés.

In [ ]:
while count >= movie_tries:  
  try:
    count -=1
    #Obtenemos el número random de 7 dígitos
    movieid = str(count).zfill(7)

    #Validamos que no esté repetido
    if movieid in lista_randoms:
      print(count, 'repetido')
      continue

    lista_randoms.append(movieid)

    #Definimos la url
    url = "http://www.omdbapi.com/?i=tt" + movieid + "&apikey=YOUR_API_KEY"

    #Hacemos el request
    response = requests.get(url)

    #Si el request jaló
    if response.status_code == 200:
      #Obtenemos el jason
      data = response.json()
      #Validación de año
      if 'Year' in data:
        #Quiero leer años mayores a 1970
        if int(ObtenerAnio(data['Year'])) < 1970:        
          print('Viejito', count, movieid, data['Year'])
          continue
      else:
        continue
      #Quiero leer solo películas
      if 'Type' in data:
        if data['Type'] != 'movie':        
          print('No película', count, movieid, data['Type'])
          continue
      else:
        continue
      #Quiero leer películas que no sean documentales, cortos o de adultos
      if 'Genre' in data:
        if 'Adult' in data['Genre'] or 'N/A' in data['Genre'] or 'Documentary' in data['Genre'] or 'Short' in data['Genre'] or 'Biography' in data['Genre']:
          print('No apto', count, movieid, data['Genre'])
          continue
      else:
        continue
      #Que tengan poster para ver
      if 'Poster' in data:
        if data['Poster'] == 'N/A':        
          print('No poster', count, movieid)
          continue
      else:
        continue
      #Validación del lenguaje
      if 'Language' in data:
        #Quiero leer solo películas en inglés
        if 'English' not in data['Language']:
          print('No inglés', count, movieid, data['Language'])
          continue
      else:
        continue
      #Si llegamos aquí, es porque admitimos a la película
      movie_counter += 1
      df_row = pd.DataFrame([data])
      movies = pd.concat([movies, df_row], ignore_index=True)
      print(data)
      print('Aceptado')
    else:
      # Error handling
      print(f"Error: {response.status_code}")
  except ValueError:
    continue

6374373 repetido
6374372 repetido
6374362 repetido
No película 6374358 6374358 episode
No apto 6374356 6374356 Adult
No película 6374354 6374354 episode
No película 6374352 6374352 episode
No película 6374348 6374348 episode
No película 6374344 6374344 episode
No película 6374342 6374342 episode
No película 6374340 6374340 episode
6374338 repetido
No película 6374336 6374336 episode
No apto 6374334 6374334 Short, Horror
6374333 repetido
No película 6374332 6374332 episode
No apto 6374330 6374330 N/A
No película 6374328 6374328 episode
No película 6374326 6374326 series
No apto 6374324 6374324 Short
No película 6374322 6374322 series
No película 6374320 6374320 episode
No apto 6374318 6374318 Short, Drama
No película 6374316 6374316 episode
No película 6374314 6374314 episode
No película 6374312 6374312 episode
{'Title': 'OverKill: The Unsolved Murder of JonBenet Ramsey', 'Year': '2016', 'Rated': 'N/A', 'Released': '17 Dec 2016', 'Runtime': 'N/A', 'Genre': 'Crime', 'Director': 'N/A', 'W

Por último, se reemplazan los csv de películas y randoms para la siguiente ejecución.

In [52]:
# Se reemplaza el archivo de películas
movies.to_csv("C:/Users/clau_/OneDrive/Documentos/FCFM/03_Tetramestre/Aprendizaje_Profundo/PIA/movies.csv", index=False)

In [53]:
# Se reemplaza el archivo de randoms
randoms = pd.DataFrame(lista_randoms, columns=['movieid'])
randoms.to_csv("C:/Users/clau_/OneDrive/Documentos/FCFM/03_Tetramestre/Aprendizaje_Profundo/PIA/lista_randoms.csv", header=False, index=False)